# Партиционирование и оптимизация Hive — 30 заданий

Практика выполняется на eBay в `/data/raw/ebay`. Решений нет.

## Результаты обучения

После **Партиционирование Hive** вы должны объяснить внутренний механизм, предсказать изменения metadata/files, выбрать безопасную команду и доказать итог измерением, а не сообщением об успехе.

## Архитектурная модель

Hive partition — каталог, зарегистрированный в Metastore. Pruning исключает каталоги только при распознаваемом предикате; высокая cardinality создаёт metadata/small-files проблему.

```text
client ── metadata RPC ──► NameNode
  │                         │ block locations
  └── data stream ──► DataNode 1 ──► DataNode 2

HiveServer2 ──► Metastore (schema/location/partitions)
      └──────► execution engine ──► HDFS files
```
NameNode не хранит содержимое файла, а Metastore не хранит строки таблицы.

## Физическая схема eBay

```text
/data/raw/ebay/
├── snapshot_dt=2026-06-24/part-....snappy.parquet
├── snapshot_dt=2026-06-25/part-....snappy.parquet
└── ...
```
Grain: наблюдение `itemid` в `snapshot_dt`. Группы колонок: карточка/цена,
иерархия категорий, продавец, география и доставка. Полная schema — в `data-catalog`.

Общий raw read-only; результаты принадлежат `/user/$HDFS_USER/hadoop_training` и личной Hive DB.

## Алгоритм исследования

1. Зафиксируйте path/URI, owner и ожидаемый объект. 2. Снимите состояние до. 3. Выполните одно изменение. 4. Проверьте exit code. 5. Измерьте namespace/files/bytes/schema/rows. 6. Повторите команду и оцените идемпотентность. 7. Сохраните evidence.

Доказывайте pruning через EXPLAIN и input paths, а дизайн — числом partitions/files/bytes.

## Типичные ошибки

- Путать локальный путь с HDFS URI.
- Делать вывод по `ls`, не проверяя blocks/bytes/schema.
- Использовать root или 777 вместо модели доступа.
- Создавать partition-каталог без Metastore или metadata без файлов.
- Считать replication резервной копией.
- Игнорировать малые файлы и цену NameNode metadata.

## Самопроверка

1. Какие metadata изменятся? 2. Где физически лежат bytes? 3. Сколько logical и physical bytes? 4. Кто может читать/писать? 5. Что произойдёт при повторе? 6. Какая независимая команда опровергнет вывод?

## Ментальная модель

Партиция — каталог, выбранный значением ключа. Pruning работает только при предикате по partition column. Слишком много мелких партиций меняет data scan на metadata bottleneck.

## Подробная теория

### 1. Partition pruning

Hive читает только каталоги партиций, если предикат явно ограничивает partition column. Функция над ключом может помешать pruning.

### 2. Кардинальность ключа

Дата часто подходит, itemid почти всегда слишком детален. Количество каталогов и файлов должно оставаться управляемым.

### 3. Статика и динамика

Static partition фиксируется в DDL вставки; dynamic берётся из данных. Ошибка режима способна создать неожиданно много каталогов.

### 4. Статистика

ANALYZE собирает row count и column statistics. Без неё стоимость JOIN и порядок таблиц оцениваются вслепую.

### 5. Распределение

DISTRIBUTE BY управляет reducer, SORT BY сортирует внутри него, ORDER BY требует глобального порядка. Bucketing полезен лишь при согласованной записи и чтении.

## Стенд

NameNode `namenode:8020`, два DataNode, HiveServer2 `hiveserver2:10000`. Личные артефакты не создаются в общем read-only raw-слое.

## Как сдаётся задание

Валидатор проверяет артефакт и JSON-доказательство. В `command` запишите фактическую команду, в `observation` — измеренный результат, в `explanation` — почему он получился. Минимальная длина защищает от пустых ответов; содержательный смысл остаётся вашей ответственностью.

In [ ]:
import json, os, subprocess, tempfile

def save_evidence(module, task, command, observation, explanation):
    user=os.environ.get("HDFS_USER", os.environ.get("HADOOP_USER_NAME", "student"))
    target=f"/user/{user}/hadoop_training/evidence/{module}/task_{task:02d}.json"
    payload={"module":module,"task":task,"command":command,"observation":observation,"explanation":explanation}
    with tempfile.NamedTemporaryFile("w",encoding="utf-8",delete=False,suffix=".json") as f:
        json.dump(payload,f,ensure_ascii=False,indent=2); local=f.name
    subprocess.run(["hdfs","dfs","-mkdir","-p",target.rsplit("/",1)[0]],check=True)
    subprocess.run(["hdfs","dfs","-put","-f",local,target],check=True)
    os.unlink(local)
    print(target)

### Задание 1. partition key

Создайте Hive-объект `pt_01` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `partition key`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 1

### Задание 2. static partition

Создайте Hive-объект `pt_02` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `static partition`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 2

### Задание 3. dynamic partition

Создайте Hive-объект `pt_03` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `dynamic partition`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 3

### Задание 4. add partition

Создайте Hive-объект `pt_04` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `add partition`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 4

### Задание 5. drop partition

Создайте Hive-объект `pt_05` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `drop partition`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 5

### Задание 6. MSCK repair

Создайте Hive-объект `pt_06` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `MSCK repair`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 6

### Задание 7. partition discovery

Создайте Hive-объект `pt_07` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `partition discovery`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 7

### Задание 8. partition pruning

Создайте Hive-объект `pt_08` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `partition pruning`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 8

### Задание 9. explain

Создайте Hive-объект `pt_09` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `explain`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 9

### Задание 10. predicate placement

Создайте Hive-объект `pt_10` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `predicate placement`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 10

### Задание 11. date partition

Создайте Hive-объект `pt_11` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `date partition`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 11

### Задание 12. multi-level partition

Создайте Hive-объект `pt_12` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `multi-level partition`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 12

### Задание 13. over-partitioning

Создайте Hive-объект `pt_13` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `over-partitioning`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 13

### Задание 14. small partitions

Создайте Hive-объект `pt_14` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `small partitions`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 14

### Задание 15. partition stats

Создайте Hive-объект `pt_15` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `partition stats`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 15

### Задание 16. table stats

Создайте Hive-объект `pt_16` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `table stats`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 16

### Задание 17. analyze

Создайте Hive-объект `pt_17` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `analyze`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 17

### Задание 18. column stats

Создайте Hive-объект `pt_18` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `column stats`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 18

### Задание 19. join plan

Создайте Hive-объект `pt_19` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `join plan`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 19

### Задание 20. map join

Создайте Hive-объект `pt_20` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `map join`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 20

### Задание 21. sort by

Создайте Hive-объект `pt_21` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `sort by`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 21

### Задание 22. distribute by

Создайте Hive-объект `pt_22` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `distribute by`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 22

### Задание 23. cluster by

Создайте Hive-объект `pt_23` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `cluster by`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 23

### Задание 24. order by

Создайте Hive-объект `pt_24` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `order by`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 24

### Задание 25. bucketing

Создайте Hive-объект `pt_25` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `bucketing`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 25

### Задание 26. bucket count

Создайте Hive-объект `pt_26` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `bucket count`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 26

### Задание 27. sampling

Создайте Hive-объект `pt_27` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `sampling`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 27

### Задание 28. compaction

Создайте Hive-объект `pt_28` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `compaction`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 28

### Задание 29. optimized eBay table

Создайте Hive-объект `pt_29` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `optimized eBay table`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 29

### Задание 30. plan audit

Создайте Hive-объект `pt_30` в личной БД. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `plan audit`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py partitioning 30

## Итог

Все 30 проверок должны возвращать PASS. Удалять чужие или raw-данные запрещено.